# 02 – GRPO: selbst erfinden
Das Modell bekommt **nur Fragen, keine Musterantworten**. Pro Frage erzeugt es mehrere Ideen; belohnt wird:
- **creative_valid** – weit weg von der naheliegenden Lösung **und** vom Judge als funktionsfähig bewertet (Originalität zählt nur, wenn's funktioniert)
- **diversity** – unterscheidet sich von den anderen Ideen derselben Runde (gegen Einheitsbrei)
- **format** – folgt dem Denkprozess · **length** – nicht ausufernd

**Vorher:** Output von 01 als Input hinzufügen (**Add Input → Your Work → Notebook 01**), Pfad in Zelle 2 prüfen.
**Kaggle:** GPU T4 x2 · Internet an · Dauer ca. 8–10 h → **Save Version → Save & Run All (Commit)**.
GPU 0 = trainiertes Modell, GPU 1 = Judge + Embeddings.

In [ ]:
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if os.path.exists("repo"):
    subprocess.run(["git", "-C", "repo", "pull", "-q"], check=True)   # immer neuester Stand
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json, re, torch

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: eingebauten Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

def generate(model, tokenizer, question, max_new_tokens=600, temperature=0.8):
    p = chat(tokenizer, [{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

def response_parts(model_name):
    if "qwen" in model_name.lower():
        return "<|im_start|>user\n", "<|im_start|>assistant\n"
    return "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
import glob
from unsloth import PatchFastRL
try:
    PatchFastRL("GRPO", FastLanguageModel)
except Exception:
    pass  # neuere Unsloth-Versionen patchen automatisch

cands = glob.glob("/kaggle/input/**/adapter_sft", recursive=True)
SFT_ADAPTER = cands[0] if cands else "adapter_sft"
print("SFT-Adapter:", SFT_ADAPTER)

G = CFG["grpo"]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER, max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
for n, p in model.named_parameters():
    if "lora_" in n:
        p.requires_grad = True
model.print_trainable_parameters()

In [ ]:
# Judge + Embeddings auf GPU 1
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
JDEV = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
j_tok = AutoTokenizer.from_pretrained(G["judge_model"]); j_tok.padding_side = "left"
judge = AutoModelForCausalLM.from_pretrained(G["judge_model"], device_map={"": JDEV}, torch_dtype=torch.float16).eval()
embedder = SentenceTransformer(G["embed_model"], device=JDEV)

JUDGE_PROMPT = """Problem: {q}

Vorgeschlagene Idee: {idea}

Bewerte NUR, ob diese Idee das Problem tatsächlich lösen kann und realistisch umsetzbar, sicher und legal ist.
Originalität spielt keine Rolle. Vage Floskeln ohne konkrete Umsetzung bekommen höchstens 3.
Antworte ausschließlich mit einer ganzen Zahl von 0 bis 10."""

@torch.no_grad()
def judge_scores(questions, ideas, bs=8):
    out = []
    for i in range(0, len(ideas), bs):
        texts = [chat(j_tok, [{"role":"user","content":JUDGE_PROMPT.format(q=q, idea=d[:1200])}], add_generation_prompt=True)
                 for q, d in zip(questions[i:i+bs], ideas[i:i+bs])]
        enc = j_tok(texts, return_tensors="pt", padding=True).to(JDEV)
        gen = judge.generate(**enc, max_new_tokens=4, do_sample=False)
        for g in gen[:, enc.input_ids.shape[1]:]:
            m = re.search(r"\d+", j_tok.decode(g, skip_special_tokens=True))
            out.append(min(10, int(m.group())) / 10 if m else 0.0)
    return out

def emb(texts):
    return embedder.encode([f"query: {t}" for t in texts], normalize_embeddings=True, convert_to_tensor=True)

print(judge_scores(["Wie hält man Brot länger frisch?"], ["Brot in ein Leinentuch wickeln und in einem Brotkasten lagern."]))

In [ ]:
# Belohnungsfunktionen
def best_idea(text):
    m = re.search(r"^Beste Idee:\s*(.+)", text, re.M | re.S)
    return (m.group(1) if m else text[-600:]).strip()

def question_of(prompt):
    m = re.search(r"user\n(.*?)<\|im_end\|>", prompt, re.S)
    return m.group(1).strip() if m else prompt[-400:]

def r_format(prompts, completions, **kw):
    out = []
    for c in completions:
        L = c.split("\n")
        tools = sum(l.startswith("Werkzeug – ") and "→ Rohidee:" in l for l in L)
        s = 0.25 * any(l.startswith("Annahmen:") for l in L)
        s += 0.25 * (3 <= tools <= 4)
        s += 0.2 * any(l.startswith("Aussortiert:") for l in L)
        s += 0.3 * any(l.startswith("Beste Idee:") for l in L)
        out.append(s)
    return out

LOG = []
def r_creative_valid(prompts, completions, obvious, **kw):
    qs = [question_of(p) for p in prompts]
    ideas = [best_idea(c) for c in completions]
    valid = judge_scores(qs, ideas)
    E_i, E_o = emb(ideas), emb([o or q for o, q in zip(obvious, qs)])
    novelty = (1 - (E_i * E_o).sum(-1)).clamp(min=0).mul(4).clamp(max=1).tolist()
    LOG.append({"valid": sum(valid)/len(valid), "novelty": sum(novelty)/len(novelty)})
    return [v * (0.4 + 0.6 * n) for v, n in zip(valid, novelty)]   # ohne Gültigkeit keine Punkte

def r_diversity(prompts, completions, **kw):
    E = emb([best_idea(c) for c in completions]); out = []
    for i, p in enumerate(prompts):
        others = [j for j, q in enumerate(prompts) if q == p and j != i]
        out.append(0.0 if not others else float((1 - (E[others] @ E[i])).mean().mul(4).clamp(0, 1)))
    return out

def r_length(prompts, completions, **kw):
    return [0.0 if len(c.split()) <= 380 else -min(1.0, (len(c.split()) - 380) / 200) for c in completions]

W = G["reward_weights"]
REWARDS = [r_format, r_creative_valid, r_diversity, r_length]
WEIGHTS = [W["format"], W["creative_valid"], W["diversity"], W["length"]]

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files="repo/data/grpo_prompts.jsonl", split="train")
ds = ds.map(lambda r: {"prompt": chat(tokenizer, [{"role":"user","content":r["prompt"]}], add_generation_prompt=True),
                       "obvious": r["obvious"]})
ds = ds.shuffle(seed=42)
print(len(ds), "Fragen")

In [ ]:
from trl import GRPOTrainer, GRPOConfig
args = GRPOConfig(
    learning_rate=G["lr"], max_steps=G["max_steps"], save_steps=G["save_steps"], save_total_limit=3,
    per_device_train_batch_size=G["batch_size"], gradient_accumulation_steps=G["grad_accum"],
    num_generations=G["num_generations"], temperature=G["temperature"],
    max_prompt_length=G["max_prompt_length"], max_completion_length=G["max_completion_length"],
    reward_weights=WEIGHTS, optim="adamw_8bit", lr_scheduler_type="cosine", warmup_steps=5,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    logging_steps=1, output_dir="checkpoints_grpo", report_to="none", seed=42)
trainer = GRPOTrainer(model=model, processing_class=tokenizer, reward_funcs=REWARDS, args=args, train_dataset=ds)

import os
last = sorted(glob.glob("checkpoints_grpo/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
trainer.train(resume_from_checkpoint=last[-1] if last else None)
# Lesen: "reward" sollte über die Zeit steigen. Unten: valid (Judge) und novelty getrennt.

In [ ]:
import pandas as pd
df = pd.DataFrame(LOG); k = max(1, len(df)//10)
print("Gültigkeit & Originalität (gleitender Schnitt):")
print(df.rolling(k).mean().iloc[::k].round(3).to_string())
# Gesund: beide steigen oder novelty steigt bei stabiler valid. Fällt valid deutlich → Modell wird "kreativ-unsinnig".

In [ ]:
FastLanguageModel.for_inference(model)
q = json.loads(open("repo/eval/business_holdout.jsonl").readline())["prompt"]
print("FRAGE (Business, nie trainiert):", q, "\n")
print(generate(model, tokenizer, q))
model.save_pretrained("adapter_grpo"); tokenizer.save_pretrained("adapter_grpo")
print("\nGespeichert: /kaggle/working/adapter_grpo")